In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)

In [2]:
df = pd.read_csv('../data/AirQualityUCI.csv', sep=';', decimal=',')

# Drop the two junk trailing columns
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print("Shape after dropping junk columns:", df.shape)
df.dtypes

Shape after dropping junk columns: (9471, 15)


Date                 str
Time                 str
CO(GT)           float64
PT08.S1(CO)      float64
NMHC(GT)         float64
C6H6(GT)         float64
PT08.S2(NMHC)    float64
NOx(GT)          float64
PT08.S3(NOx)     float64
NO2(GT)          float64
PT08.S4(NO2)     float64
PT08.S5(O3)      float64
T                float64
RH               float64
AH               float64
dtype: object

In [4]:

print("Duplicate rows:", df.duplicated().sum())


print("Fully-empty rows:", df.isna().all(axis=1).sum())


df = df.dropna(how='all')
print("Shape after dropping empty rows:", df.shape)

Duplicate rows: 113
Fully-empty rows: 114
Shape after dropping empty rows: (9357, 15)


In [5]:

time_fixed = df['Time'].str.replace('.', ':', regex=False)

df['datetime'] = pd.to_datetime(df['Date'] + ' ' + time_fixed, format='%d/%m/%Y %H:%M:%S')

df['hour'] = df['datetime'].dt.hour
df['day'] = df['datetime'].dt.day
df['month'] = df['datetime'].dt.month
df['weekday'] = df['datetime'].dt.day_name()
df['is_weekend'] = df['datetime'].dt.dayofweek >= 5

df[['datetime', 'hour', 'day', 'month', 'weekday', 'is_weekend']].head()

,datetime,hour,day,month,weekday,is_weekend
0,2004-03-10 18:00:00,18,10,3,Wednesday,False
1,2004-03-10 19:00:00,19,10,3,Wednesday,False
2,2004-03-10 20:00:00,20,10,3,Wednesday,False
3,2004-03-10 21:00:00,21,10,3,Wednesday,False
4,2004-03-10 22:00:00,22,10,3,Wednesday,False


In [6]:
print("Is already sorted by time:", df['datetime'].is_monotonic_increasing)

df = df.sort_values('datetime').reset_index(drop=True)
print("Date range:", df['datetime'].min(), "to", df['datetime'].max())
print("Total days covered:", (df['datetime'].max() - df['datetime'].min()).days)

Is already sorted by time: True
Date range: 2004-03-10 18:00:00 to 2005-04-04 14:00:00
Total days covered: 389


In [7]:

rename_map = {
    'CO(GT)': 'CO_GT', 'PT08.S1(CO)': 'PT08_S1_CO', 'NMHC(GT)': 'NMHC_GT',
    'C6H6(GT)': 'C6H6_GT', 'PT08.S2(NMHC)': 'PT08_S2_NMHC', 'NOx(GT)': 'NOx_GT',
    'PT08.S3(NOx)': 'PT08_S3_NOx', 'NO2(GT)': 'NO2_GT', 'PT08.S4(NO2)': 'PT08_S4_NO2',
    'PT08.S5(O3)': 'PT08_S5_O3', 'T': 'Temperature', 'RH': 'RelativeHumidity', 'AH': 'AbsoluteHumidity'
}
df = df.rename(columns=rename_map)

numeric_cols = list(rename_map.values())


sentinel_counts = (df[numeric_cols] == -200).sum().sort_values(ascending=False)
sentinel_counts

NMHC_GT             8443
CO_GT               1683
NO2_GT              1642
NOx_GT              1639
PT08_S1_CO           366
PT08_S2_NMHC         366
C6H6_GT              366
PT08_S3_NOx          366
PT08_S4_NO2          366
PT08_S5_O3           366
Temperature          366
RelativeHumidity     366
AbsoluteHumidity     366
dtype: int64

In [ ]:
df[numeric_cols] = df[numeric_cols].replace(-200, np.nan)


df[numeric_cols].isna().sum().sort_values(ascending=False)

NMHC_GT             8443
CO_GT               1683
NO2_GT              1642
NOx_GT              1639
PT08_S1_CO           366
PT08_S2_NMHC         366
C6H6_GT              366
PT08_S3_NOx          366
PT08_S4_NO2          366
PT08_S5_O3           366
Temperature          366
RelativeHumidity     366
AbsoluteHumidity     366
dtype: int64

In [9]:
# Drop NMHC_GT - too sparse (90% missing) to impute credibly
df = df.drop(columns=['NMHC_GT'])
clean_cols = [c for c in numeric_cols if c != 'NMHC_GT']

# Time-based interpolation, but only across small gaps (max 6 hours)
# Longer gaps get a fallback below rather than stretching interpolation too far
df = df.set_index('datetime')
for col in clean_cols:
    df[col] = df[col].interpolate(method='time', limit=6, limit_direction='both')
df = df.reset_index()

print("Remaining missing after interpolation:")
print(df[clean_cols].isna().sum().sort_values(ascending=False))

Remaining missing after interpolation:
CO_GT               1181
NO2_GT              1064
NOx_GT              1062
PT08_S1_CO           240
PT08_S2_NMHC         240
C6H6_GT              240
PT08_S3_NOx          240
PT08_S4_NO2          240
PT08_S5_O3           240
Temperature          240
RelativeHumidity     240
AbsoluteHumidity     240
dtype: int64


In [10]:
for col in clean_cols:
    if df[col].isna().any():
        hourly_median = df.groupby('hour')[col].transform('median')
        df[col] = df[col].fillna(hourly_median)

print("Remaining missing after fallback:", df[clean_cols].isna().sum().sum())

Remaining missing after fallback: 0


In [11]:
stats_rows = []
for col in clean_cols:
    values = df[col].dropna().to_numpy()
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    n_outliers = int(((df[col] < lower_bound) | (df[col] > upper_bound)).sum())

    stats_rows.append({
        'column': col,
        'mean': np.mean(values),
        'median': np.median(values),
        'std': np.std(values, ddof=1),
        'p10': np.percentile(values, 10),
        'p90': np.percentile(values, 90),
        'iqr_lower_bound': lower_bound,
        'iqr_upper_bound': upper_bound,
        'n_outliers': n_outliers
    })

stats_table = pd.DataFrame(stats_rows)
stats_table.round(2)

,column,mean,median,std,p10,p90,iqr_lower_bound,iqr_upper_bound,n_outliers
0,CO_GT,2.11,1.80,1.40,0.60,3.91,-1.45,5.35,301
1,PT08_S1_CO,1100.08,1067.00,215.35,852.00,1408.00,501.50,1665.50,125
2,C6H6_GT,10.11,8.30,7.44,2.50,20.30,-9.75,28.25,249
3,PT08_S2_NMHC,940.46,912.00,266.12,621.00,1304.00,164.50,1688.50,68
4,NOx_GT,234.56,177.00,200.02,53.43,504.40,-201.50,594.50,619
5,PT08_S3_NOx,834.65,803.00,253.84,538.00,1150.00,202.00,1426.00,248
6,NO2_GT,110.54,109.00,46.36,55.00,172.00,-15.50,228.50,152
7,PT08_S4_NO2,1456.13,1465.00,341.39,990.00,1886.00,597.50,2305.50,122
8,PT08_S5_O3,1024.90,966.00,396.77,549.60,1586.40,-70.50,2077.50,100
9,Temperature,18.31,17.70,8.73,7.00,30.10,-6.30,42.50,9


In [12]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df['pollution_index'] = (
    normalize(df['CO_GT']) + normalize(df['NOx_GT']) + normalize(df['NO2_GT'])
) / 3

df['pollution_index'].describe()

count    9357.000000
mean        0.216180
std         0.119168
min         0.005783
25%         0.128216
50%         0.198889
75%         0.275832
max         0.873483
Name: pollution_index, dtype: float64

In [13]:
valid_rows = df[['PT08_S1_CO', 'CO_GT']].dropna()
sensor_reliability_ratio = valid_rows['PT08_S1_CO'] / valid_rows['CO_GT'].replace(0, np.nan)

print(sensor_reliability_ratio.describe())

count     9357.000000
mean       806.852677
std        825.227553
min        129.302326
25%        429.642857
50%        593.888889
75%        902.500000
max      11270.000000
dtype: float64
